# Figure 2 — does the signature generalize?

The central claim: signal response signatures transfer across batches, cell
types and species. Four panels:

* **2c** leave-one-screen-out cross-validation
* **2d** endoderm → mesoderm (developmentally non-interconvertible)
* **2e** IRIS vs classical ML
* **2f** gene ablation — how distributed is the signature?

Script equivalents live in `figures/fig2/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "iris_repro").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "figures"))

import numpy as np
import pandas as pd
from iris_repro import config, data, metrics, plotting, provenance

plt = plotting.set_style()
OUT = config.output_dir("fig2")
print("outputs ->", OUT)

## 2c — hold out one screen at a time

Predictions come from the saved runner CSVs. Only *held-out* rows are scored;
`clean_splits2` supersedes `clean_splits` where a split was rerun.

In [ ]:
from fig2.fig2cd_generalization import collect, score_splits, make_panel

cv = collect("clean_splits", single_holdout_only=True)
print(f"{len(cv):,} held-out cells across {cv['source'].nunique()} runs")
cv_table, cv_curves = score_splits(cv)
cv_table.round(3)

In [ ]:
cv_table.groupby("signal")[["AUROC", "AUPRC", "F1"]].mean().round(3)

Each dot is one pathway in one held-out screen; above the diagonal beats chance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(4.6, 2.2))
plotting.plot_f1_vs_baseline(cv_table, ax=axes[0])
axes[0].set_title("Cross-validation")
plotting.plot_pr(cv_curves, ax=axes[1], title="Pooled PR")
plt.tight_layout()
plt.show()

## 2d — endoderm to mesoderm

Trained only on the endodermal screens (mE_d2 + hE_d8), tested on the
mesodermal ones. The two lineages are distinct and non-interconvertible, so
nothing about mesoderm is in the training set.

In [ ]:
endo_train = tuple(config.load_config()["splits"]["endoderm_to_mesoderm"]["train"])
e2m = collect("clean_splits", train=endo_train)
e2m_table, e2m_curves = score_splits(e2m)
e2m_table.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(4.6, 2.2))
plotting.plot_f1_vs_baseline(e2m_table, ax=axes[0])
axes[0].set_title("Endoderm to mesoderm")
plotting.plot_pr(e2m_curves, ax=axes[1], title="Pooled PR")
plt.tight_layout()
plt.show()

e2m_table.groupby("signal")[["AUROC", "AUPRC", "F1"]].mean().round(3)

WNT and RA transfer best; FGF is weakest — consistent with the Discussion,
which attributes FGF's behaviour to signal-transduction components shared with
other receptor tyrosine kinase pathways.

## 2f — gene ablation

Genes are ranked (mutual information, dispersion, or expression) and
progressively randomized by resampling with replacement. If a few canonical
genes carried the signal, accuracy would collapse immediately.

In [ ]:
from fig2.fig2f_gene_ablation import (load_ablation_results, normalize_curve,
                                      fit_exponential, genes_to_half,
                                      GENES_PER_BLOCK)

print(f"ablation block size: {GENES_PER_BLOCK} genes")
abl = load_ablation_results("out")
abl.head()

In [ ]:
rows = []
for (sig, assay, held), grp in abl.groupby(["signal", "assay", "held_out"]):
    curve = normalize_curve(grp)
    rows.append({"signal": sig, "assay": assay, "held_out": held,
                 "genes_to_50pct": genes_to_half(curve, "fit")})
half = pd.DataFrame(rows)

(half[half["assay"] == "mi"]
   .pivot(index="held_out", columns="signal", values="genes_to_50pct")
   .round(0))

Thousands of genes must be destroyed before performance halves: the signature is transcriptome-wide.

In [ ]:
mi = abl[abl["assay"] == "mi"]
colors = config.palette()
sigs = [s for s in config.signals() if s in set(mi["signal"])]

fig, axes = plt.subplots(1, len(sigs), figsize=(1.5 * len(sigs), 1.7), sharey=True)
for ax, sig in zip(np.atleast_1d(axes), sigs):
    for held, grp in mi[mi["signal"] == sig].groupby("held_out"):
        curve = normalize_curve(grp)
        a, ok = fit_exponential(curve)
        ax.plot(curve["n_ablated"], curve["AUROC_norm"],
                color=colors.get(sig, "k"), lw=0.4, alpha=0.25)
        if ok:
            xs = np.linspace(0, curve["n_ablated"].max(), 200)
            ax.plot(xs, np.exp(-a * xs), color=colors.get(sig, "k"), lw=0.9)
    ax.set_title(config.display_name(sig))
    ax.set_ylim(0, 1.02)
np.atleast_1d(axes)[0].set_ylabel("Normalized AUROC")
fig.supxlabel("Genes randomized (ranked by mutual information)", fontsize=7)
plt.tight_layout()
plt.show()